In [3]:
#pip install jupyter pandas scikit-learn

In [4]:
import pandas as pd

df = pd.read_csv('../data/all_data.csv')
df_eng = df[df['Review_Language'] == 'English']
df_eng.to_csv('../data/data_eng.csv')

In [5]:
len(df_eng)

2000

In [6]:
df = pd.read_csv('../data/data_eng.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review
0,985,0,English,Ankara,Hotel Ickale,Staff. Serkan türk was a great help during our...,NaN,10.0,POS,0,NaN,False,True
1,986,1,English,Ankara,Hotel Ickale,Nicely furnished room and nicely decorated lob...,NaN,9.0,POS,0,NaN,False,True
2,987,2,English,Ankara,Hotel Ickale,"Very central, near 2 metro stations. Room was ...",The wifi had annoying login that required logg...,9.0,POS,0,NaN,False,False
3,988,3,English,Ankara,Hotel Ickale,Lovely interior.. great location.. friendly staff,what is not to like\r\nrestaurant staff could ...,9.0,POS,0,NaN,False,False
4,989,4,English,Ankara,Hotel Ickale,"Thanks to Mr.Serkan, our problems were solved",NaN,9.0,POS,0,NaN,False,True


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/data_eng.csv')

# label: source column, check which value is fake
print(df['source'].value_counts())
print(df.groupby('source')['Prompt_Language'].apply(lambda s: s.notna().sum()))

# text: combine upside and downside review, missing values become empty strings
df['text'] = df['Upside_Review'].fillna('') + ' ' + df['Downside_Review'].fillna('')
df['text'] = df['text'].str.strip()

# stratified 75/25 split, whole rows kept so Sentiment stays aligned for question 3
train_df, test_df = train_test_split(
    df, test_size=0.25, stratify=df['source'], random_state=42
)

print(len(train_df), len(test_df))
print(train_df['source'].value_counts())
print(test_df['source'].value_counts())

train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)


source
0    1000
1    1000
Name: count, dtype: int64
source
0       0
1    1000
Name: Prompt_Language, dtype: int64
1500 500
source
0    750
1    750
Name: count, dtype: int64
source
1    250
0    250
Name: count, dtype: int64


In [8]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

# empty texts are read back from csv as NaN
train_df['text'] = train_df['text'].fillna('')
test_df['text'] = test_df['text'].fillna('')

y_train = train_df['source']
y_test = test_df['source']

# bag-of-words: lowercase, English stop words removed, words in fewer than 5 training reviews dropped
vec = CountVectorizer(lowercase=True, stop_words='english', min_df=5)

# vocabulary learned on the training set only
X_train = vec.fit_transform(train_df['text'])

# test set uses the training vocabulary, no refitting
X_test = vec.transform(test_df['text'])

print(X_train.shape)
print(X_test.shape)
print(len(vec.get_feature_names_out()))
print(vec.get_feature_names_out()[:20])

(1500, 1165)
(500, 1165)
1165
['10' '15' '20' '24' '30' '40' '50' 'able' 'absolute' 'absolutely' 'ac'
 'acceptable' 'access' 'accessible' 'accommodate' 'accommodating'
 'accommodation' 'accommodations' 'actual' 'actually']


In [9]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

train_df['text'] = train_df['text'].fillna('')
test_df['text'] = test_df['text'].fillna('')

# check for literal backslash-n sequences
print(train_df['text'].str.contains(r'\\n', regex=True).sum())

# replace literal \n and \r with a space
train_df['text'] = train_df['text'].str.replace(r'\\[nr]', ' ', regex=True)
test_df['text'] = test_df['text'].str.replace(r'\\[nr]', ' ', regex=True)

y_train = train_df['source']
y_test = test_df['source']

# token_pattern: words of 2+ letters only, so numbers are excluded
vec = CountVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=5,
    token_pattern=r'(?u)\b[^\W\d_]{2,}\b',
)

X_train = vec.fit_transform(train_df['text'])
X_test = vec.transform(test_df['text'])

print(X_train.shape)
print(X_test.shape)
print(vec.get_feature_names_out()[:20])

0
(1500, 1158)
(500, 1158)
['able' 'absolute' 'absolutely' 'ac' 'acceptable' 'access' 'accessible'
 'accommodate' 'accommodating' 'accommodation' 'accommodations' 'actual'
 'actually' 'add' 'added' 'addition' 'additional' 'additionally' 'address'
 'addressed']


In [10]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# same 5 folds for every cross-validated model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. multinomial naive Bayes: k top features by chi-squared
nb_cv = GridSearchCV(
    Pipeline([("sel", SelectKBest(chi2)), ("clf", MultinomialNB())]),
    {"sel__k": [50, 100, 250, 500, 800, "all"]},
    cv=cv, scoring="accuracy",
)
nb_cv.fit(X_train, y_train)
print("NB:", nb_cv.best_params_, round(nb_cv.best_score_, 4))

# 2. logistic regression with Lasso penalty: C = 1/lambda
lr_cv = GridSearchCV(
    LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000),
    {"C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
    cv=cv, scoring="accuracy",
)
lr_cv.fit(X_train, y_train)
print("LR:", lr_cv.best_params_, round(lr_cv.best_score_, 4))

# 3. single classification tree: cost-complexity pruning parameter alpha
tree_cv = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {"ccp_alpha": [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]},
    cv=cv, scoring="accuracy",
)
tree_cv.fit(X_train, y_train)
print("Tree:", tree_cv.best_params_, round(tree_cv.best_score_, 4))

# 4. random forest: out-of-bag accuracy instead of cross-validation
rf_results = []
for n_trees in [200, 500]:
    for mf in ["sqrt", "log2", 0.1]:
        rf = RandomForestClassifier(
            n_estimators=n_trees, max_features=mf, oob_score=True, random_state=42, n_jobs=-1
        )
        rf.fit(X_train, y_train)
        rf_results.append((rf.oob_score_, n_trees, mf))
        print("RF", n_trees, mf, round(rf.oob_score_, 4))
best_oob, best_n, best_mf = max(rf_results, key=lambda r: r[0])
print("RF best:", best_n, best_mf, round(best_oob, 4))

# 5. gradient boosting: learning rate and tree depth
gb_cv = GridSearchCV(
    GradientBoostingClassifier(n_estimators=300, random_state=42),
    {"learning_rate": [0.03, 0.1, 0.3], "max_depth": [1, 2, 3]},
    cv=cv, scoring="accuracy", n_jobs=-1,
)
gb_cv.fit(X_train, y_train)
print("GB:", gb_cv.best_params_, round(gb_cv.best_score_, 4))

NB: {'sel__k': 500} 0.9053


C:\Users\Bobby Zafran\PycharmProjects\datamining\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Bobby Zafran\PycharmProjects\datamining\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\Bobby Zafran\PycharmProjects\datamining\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

LR: {'C': 1} 0.902
Tree: {'ccp_alpha': 0.002} 0.798
RF 200 sqrt 0.8973
RF 200 log2 0.9053
RF 200 0.1 0.888
RF 500 sqrt 0.9013
RF 500 log2 0.914
RF 500 0.1 0.882
RF best: 500 log2 0.914
GB: {'learning_rate': 0.3, 'max_depth': 3} 0.9113
